# **1. Prepare the datasets**

**1.1. Load data from HF**

In [ ]:
pip install langchain-huggingface

In [ ]:
from datasets import load_dataset
from collections import defaultdict

data = load_dataset("hoangphuc090104/DCR_Energy_Plan")["train"]

print(f"Number of rows in data: {len(data)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


energy_plans.csv:   0%|          | 0.00/212M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25670 [00:00<?, ? examples/s]

Number of rows in data: 25670


**1.2. Preproccesing data (Drop NULL and singleton columns)**

In [ ]:
# Remove NULL columns
null_cols = [c for c in data.column_names if any(v is None for v in data[c])]
print(f"Removing NULL columns: {null_cols}")
data = data.remove_columns(null_cols)

# Remove single-value columns
single_value_cols = [c for c in data.column_names if len(set(data[c])) == 1]
print(f"Removing single-value columns: {single_value_cols}")
data = data.remove_columns(single_value_cols)

print("Done. All NULL and Singleton columns dropped")

Removing NULL columns: ['factsheet_url', 'effective_to', 'pcr', 'compliance_score', 'created_by', 'updated_by', 'postcode']
Removing single-value columns: ['fuel_type', 'compliance_checklist', 'ops_status']
Done. All NULL and Singleton columns dropped


**1.3. Convert to 3 types: hart attributes, soft attributes and cost attributes**

In [ ]:
import json
import re


def safe_json_loads(value, default=None):
    if default is None:
        default = {}

    if isinstance(value, dict):
        return value

    if isinstance(value, list):
        return value

    if not isinstance(value, str):
        return default

    try:
        return json.loads(value)
    except Exception:
        return default


def has_non_empty_list(value):
    return isinstance(value, list) and len(value) > 0


def convert_numeric_strings_to_float(obj):
    if isinstance(obj, dict):
        return {k: convert_numeric_strings_to_float(v) for k, v in obj.items()}

    if isinstance(obj, list):
        return [convert_numeric_strings_to_float(elem) for elem in obj]

    if isinstance(obj, str):
        try:
            return float(obj)
        except ValueError:
            return obj

    return obj


def extract_tariff_type(pricing_model):
    if pricing_model in ["SINGLE_RATE", "SINGLE_RATE_CONT_LOAD"]:
        return "SINGLE_RATE"

    if pricing_model in ["TIME_OF_USE", "TIME_OF_USE_CONT_LOAD"]:
        return "TIME_OF_USE"

    return None


def should_skip_plan(pricing_model):
    return pricing_model in ["FLEXIBLE", "FLEXIBLE_CONT_LOAD"]


def tariff_type_to_text(tariff_type):
    mapping = {
        "SINGLE_RATE": "Single-rate electricity plan with general anytime usage.",
        "TIME_OF_USE": "Time-of-use electricity plan with peak, shoulder, and off-peak usage periods."
    }

    return mapping.get(tariff_type, "")


EV_PATTERNS = [
    r"\bev\b",
    r"electric vehicle",
    r"electric vehicles",
    r"electric car",
    r"ev charging",
    r"ev owner",
    r"ev owners",
    r"own an ev",
    r"homeev",
    r"ev registration"
]


def contains_ev_keyword(text):
    if not text:
        return False

    text = str(text).lower()

    for pattern in EV_PATTERNS:
        if re.search(pattern, text):
            return True

    return False


def extract_has_ev(contract, plan_name=""):
    if contains_ev_keyword(plan_name):
        return True

    fields_to_check = [
        contract.get("terms", ""),
        contract.get("onExpiryDescription", ""),
        contract.get("additionalFeeInformation", "")
    ]

    for text in fields_to_check:
        if contains_ev_keyword(text):
            return True

    for field in ["eligibility", "incentives"]:
        items = contract.get(field, [])

        if not isinstance(items, list):
            continue

        for item in items:
            item_text = json.dumps(item, ensure_ascii=False)
            if contains_ev_keyword(item_text):
                return True

    return False


def build_soft_text(plan, contract, tariff_type, has_controlled_load, has_solar, has_ev):
    parts = {}

    plan_name = plan.get("display_name") or ""
    retailer_name = plan.get("brand_name") or ""

    parts["plan_name"] = plan_name

    if retailer_name:
        parts["retailer_text"] = f"Energy plan from {retailer_name}."
    else:
        parts["retailer_text"] = ""

    parts["tariff_type_text"] = tariff_type_to_text(tariff_type)

    contract_parts = []

    term_type = contract.get("termType")
    benefit_period = contract.get("benefitPeriod")
    terms = contract.get("terms")
    variation = contract.get("variation")

    if term_type:
        contract_parts.append(f"Contract term type: {term_type}.")

    if benefit_period:
        contract_parts.append(f"Benefit period: {benefit_period}.")

    if terms:
        contract_parts.append(terms)

    if variation:
        contract_parts.append(variation)

    parts["contract_text"] = " ".join(contract_parts).strip()

    billing_parts = []

    bill_frequency = contract.get("billFrequency", [])
    payment_options = contract.get("paymentOption", [])

    if bill_frequency:
        billing_parts.append(f"Bill frequency options: {', '.join(bill_frequency)}.")

    if payment_options:
        billing_parts.append(f"Payment options: {', '.join(payment_options)}.")

    parts["billing_text"] = " ".join(billing_parts).strip()

    tariff_parts = [tariff_type_to_text(tariff_type)]

    tariff_periods = contract.get("tariffPeriod", [])

    for period in tariff_periods:
        single_rate = period.get("singleRate")

        if isinstance(single_rate, dict):
            display_name = single_rate.get("displayName")
            description = single_rate.get("description")

            if display_name:
                tariff_parts.append(f"Single-rate usage: {display_name}.")
            if description:
                tariff_parts.append(f"Usage description: {description}.")

        tou_rates = period.get("timeOfUseRates", [])

        if isinstance(tou_rates, list):
            tou_parts = []

            for item in tou_rates:
                rate_type = item.get("type")
                display_name = item.get("displayName")
                description = item.get("description")

                text = rate_type or display_name

                if description:
                    text = f"{text}: {description}"

                if text:
                    tou_parts.append(text)

            if tou_parts:
                tariff_parts.append(
                    "Time-of-use periods include " + "; ".join(tou_parts) + "."
                )

    parts["tariff_structure_text"] = " ".join(tariff_parts).strip()

    if has_controlled_load:
        controlled_load_parts = []

        controlled_load = contract.get("controlledLoad", [])

        if isinstance(controlled_load, list):
            for item in controlled_load:
                display_name = item.get("displayName")
                single_rate = item.get("singleRate", {})

                description = single_rate.get("description")
                rate_display_name = single_rate.get("displayName")

                text_parts = []

                if display_name:
                    text_parts.append(display_name)

                if rate_display_name and rate_display_name != display_name:
                    text_parts.append(rate_display_name)

                if description:
                    text_parts.append(description)

                if text_parts:
                    controlled_load_parts.append(
                        "Controlled load support: " + ". ".join(text_parts) + "."
                    )

        parts["controlled_load_text"] = " ".join(controlled_load_parts).strip()
    else:
        parts["controlled_load_text"] = ""

    if has_solar:
        solar_parts = []

        solar_tariffs = contract.get("solarFeedInTariff", [])

        if isinstance(solar_tariffs, list):
            for tariff in solar_tariffs:
                display_name = tariff.get("displayName")
                description = tariff.get("description")

                if description:
                    solar_parts.append(description)
                elif display_name:
                    solar_parts.append(display_name)

        if solar_parts:
            parts["solar_text"] = "Solar feed-in tariff support: " + " ".join(solar_parts)
        else:
            parts["solar_text"] = "Solar feed-in tariff support is available."
    else:
        parts["solar_text"] = ""

    if has_ev:
        parts["ev_text"] = "This plan includes EV-related eligibility, incentives, or plan features."
    else:
        parts["ev_text"] = ""

    parts["expiry_text"] = contract.get("onExpiryDescription") or ""
    parts["additional_info_text"] = contract.get("additionalFeeInformation") or ""

    return parts


def transform_plan(plan):
    geography = safe_json_loads(plan.get("geography"), default={})
    contract = safe_json_loads(plan.get("electricity_contract"), default={})

    pricing_model = contract.get("pricingModel")

    if should_skip_plan(pricing_model):
        return None

    tariff_type = extract_tariff_type(pricing_model)

    if tariff_type is None:
        return None

    has_controlled_load = has_non_empty_list(contract.get("controlledLoad"))
    has_solar = has_non_empty_list(contract.get("solarFeedInTariff"))
    has_ev = extract_has_ev(
        contract=contract,
        plan_name=plan.get("display_name", "")
    )

    hard_attributes = {
        "retailer_name": plan.get("brand_name"),
        "customer_type": plan.get("customer_type"),
        "distributors": geography.get("distributors", []),
        "included_postcodes": geography.get("includedPostcodes", []),
        "tariff_type": tariff_type,
        "has_controlled_load": has_controlled_load,
        "has_solar": has_solar,
        "has_ev": has_ev
    }

    soft_text = build_soft_text(
        plan=plan,
        contract=contract,
        tariff_type=tariff_type,
        has_controlled_load=has_controlled_load,
        has_solar=has_solar,
        has_ev=has_ev
    )

    full_text = " ".join(
        value.strip()
        for value in soft_text.values()
        if isinstance(value, str) and value.strip()
    )

    return {
        "plan_id": plan.get("plan_id"),
        "hard_attributes": hard_attributes,
        "soft_text": soft_text,
        "full_text": full_text
    }

In [ ]:
def transform_row(row):
    processed_plan = transform_plan(row)

    if processed_plan is None:
        return {
            "processed_plan": None
        }

    return {
        "processed_plan": convert_numeric_strings_to_float(processed_plan)
    }


processed_data = data.map(
    transform_row,
    remove_columns=data.column_names
)

processed_data = processed_data.filter(
    lambda row: row["processed_plan"] is not None
)

Map:   0%|          | 0/25670 [00:00<?, ? examples/s]

Filter:   0%|          | 0/25670 [00:00<?, ? examples/s]

In [ ]:
print(f"Number of rows in data: {len(processed_data)}")
print(processed_data[0]["processed_plan"]["hard_attributes"])
print(data[0])

Number of rows in data: 25333
{'customer_type': 'RESIDENTIAL', 'distributors': ['Ausgrid'], 'has_controlled_load': False, 'has_ev': False, 'has_solar': True, 'included_postcodes': [2000.0, 2007.0, 2008.0, 2009.0, 2010.0, 2011.0, 2015.0, 2016.0, 2017.0, 2018.0, 2019.0, 2020.0, 2021.0, 2022.0, 2023.0, 2024.0, 2025.0, 2026.0, 2027.0, 2028.0, 2029.0, 2030.0, 2031.0, 2032.0, 2033.0, 2034.0, 2035.0, 2036.0, 2037.0, 2038.0, 2039.0, 2040.0, 2041.0, 2042.0, 2043.0, 2044.0, 2045.0, 2046.0, 2047.0, 2048.0, 2049.0, 2050.0, 2060.0, 2061.0, 2062.0, 2063.0, 2064.0, 2065.0, 2066.0, 2067.0, 2068.0, 2069.0, 2070.0, 2071.0, 2072.0, 2073.0, 2074.0, 2075.0, 2076.0, 2077.0, 2079.0, 2080.0, 2081.0, 2082.0, 2083.0, 2084.0, 2085.0, 2086.0, 2087.0, 2088.0, 2089.0, 2090.0, 2092.0, 2093.0, 2094.0, 2095.0, 2096.0, 2097.0, 2099.0, 2100.0, 2101.0, 2102.0, 2103.0, 2104.0, 2105.0, 2106.0, 2107.0, 2108.0, 2110.0, 2111.0, 2112.0, 2113.0, 2114.0, 2118.0, 2119.0, 2120.0, 2121.0, 2122.0, 2125.0, 2126.0, 2127.0, 2128.0, 213

**1.4. Batch Plan**

In [ ]:
import os
import json
import time
import math
import subprocess
from sentence_transformers import SentenceTransformer

BUCKET = "energy-plan-bucket-1"
GCS_PREFIX = "gte-modernbert-processed-plans"
MODEL_NAME = "Alibaba-NLP/gte-modernbert-base"

MAX_ROWS = 25000
BATCH_SIZE = 1000
EMBED_BATCH_SIZE = 16
VECTOR_DIMENSION = 768

TOTAL_ROWS = min(len(processed_data), MAX_ROWS)
NUM_BATCHES = math.ceil(TOTAL_ROWS / BATCH_SIZE)

model = SentenceTransformer(MODEL_NAME, device="cuda")

print("Total rows:", TOTAL_ROWS)
print("Batch size:", BATCH_SIZE)
print("Number of batches:", NUM_BATCHES)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/205 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Total rows: 25000
Batch size: 1000
Number of batches: 25


# 2. Upload vector to vertex AI Vector Search

**2.1. Define Helper Function**

In [ ]:
def normalize_restrict_value(value):
    if value is None:
        return None

    if isinstance(value, bool):
        return str(value).lower()

    if isinstance(value, float):
        if value.is_integer():
            return str(int(value))
        return str(value)

    if isinstance(value, int):
        return str(value)

    text = str(value).strip()

    if not text:
        return None

    try:
        number = float(text)
        if number.is_integer():
            return str(int(number))
    except:
        pass

    return text


def gcs_exists(path):
    result = subprocess.run(
        ["gsutil", "ls", path],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    return result.returncode == 0


def run_cmd(cmd):
    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.stdout:
        print(result.stdout)

    if result.stderr:
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")


def get_processed_plan(row):
    value = row.get("processed_plan")

    if isinstance(value, str):
        return json.loads(value)

    if isinstance(value, dict):
        return value

    return row


def as_allow_list(value):
    if value is None:
        return []

    if isinstance(value, list):
        result = []

        for v in value:
            normalized = normalize_restrict_value(v)

            if normalized is not None:
                result.append(normalized)

        return sorted(set(result))

    normalized = normalize_restrict_value(value)

    return [normalized] if normalized is not None else []


def build_restricts(hard_attributes):
    restricts = []

    fields = [
        "retailer_name",
        "customer_type",
        "distributors",
        "included_postcodes",
        "tariff_type",
        "has_controlled_load",
        "has_solar",
        "has_ev"
    ]

    for field in fields:
        value = hard_attributes.get(field)
        allow_values = as_allow_list(value)

        if not allow_values:
            continue

        restricts.append({
            "namespace": field,
            "allow": allow_values
        })

    return restricts


def upload_batch(batch_idx):
    if batch_idx < 0 or batch_idx >= NUM_BATCHES:
        raise ValueError(f"batch_idx must be between 0 and {NUM_BATCHES - 1}")

    start = batch_idx * BATCH_SIZE
    end = min(start + BATCH_SIZE, TOTAL_ROWS)

    batch_name = f"batch_{batch_idx:03d}.json"
    manifest_name = f"manifest_{batch_idx:03d}.json"

    local_batch_path = batch_name
    local_manifest_path = manifest_name

    gcs_batch_path = f"gs://{BUCKET}/{GCS_PREFIX}/batches/{batch_name}"
    gcs_manifest_path = f"gs://{BUCKET}/{GCS_PREFIX}/manifests/{manifest_name}"

    print(f"Checking batch {batch_idx}: rows {start} to {end}")

    if gcs_exists(gcs_batch_path):
        print(f"Batch {batch_idx} already exists on GCS. Skipping.")
        return

    batch_data = processed_data.select(range(start, end))

    plans = []
    texts = []

    for row in batch_data:
        processed_plan = get_processed_plan(row)
        full_text = processed_plan.get("full_text", "")

        if not full_text:
            continue

        plans.append(processed_plan)
        texts.append(full_text)

    print(f"Valid plans in batch {batch_idx}: {len(plans)}")

    if not texts:
        print(f"No valid text found in batch {batch_idx}. Skipping.")
        return

    print(f"Encoding batch {batch_idx}")

    encode_start = time.time()

    vectors = model.encode(
        texts,
        batch_size=EMBED_BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    encode_end = time.time()

    print(f"Encoding completed in {encode_end - encode_start:.2f} seconds")

    print(f"Writing {batch_name}")

    with open(local_batch_path, "w", encoding="utf-8") as f:
        for i, vec in enumerate(vectors):
            plan = plans[i]

            plan_id = str(plan.get("plan_id", f"row_{start + i}"))
            hard_attributes = plan.get("hard_attributes", {})

            record = {
                "id": plan_id,
                "embedding": vec.tolist(),
                "restricts": build_restricts(hard_attributes)
            }

            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    manifest = {
        "batch_idx": batch_idx,
        "batch_name": batch_name,
        "model": MODEL_NAME,
        "dimension": VECTOR_DIMENSION,
        "normalize_embeddings": True,
        "start_index": start,
        "end_index": end,
        "num_items": len(plans),
        "gcs_batch_path": gcs_batch_path,
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S")
    }

    with open(local_manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)

    print(f"Uploading {batch_name} to GCS")
    run_cmd(["gsutil", "cp", local_batch_path, gcs_batch_path])

    print(f"Uploading {manifest_name} to GCS")
    run_cmd(["gsutil", "cp", local_manifest_path, gcs_manifest_path])

    os.remove(local_batch_path)
    os.remove(local_manifest_path)

    print(f"Batch {batch_idx} uploaded successfully")

In [ ]:
sample_plan = get_processed_plan(processed_data[0])

print(json.dumps(
    sample_plan["hard_attributes"],
    indent=2,
    ensure_ascii=False
))

print(json.dumps(
    build_restricts(sample_plan["hard_attributes"]),
    indent=2,
    ensure_ascii=False
))

**2.2. Set up Cloud Storage**

In [ ]:
# Step 1: Authentication
from google.colab import auth
auth.authenticate_user()

# Step 2: Connect GC Console
PROJECT_ID = "project-ce1ff6dc-7e15-4f39-bb3"
!gcloud config set project $PROJECT_ID

# Step 3: Enable services
!gcloud services enable \
aiplatform.googleapis.com \
vectorsearch.googleapis.com

# Step 4: Init vertex AI

!pip install -q google-cloud-aiplatform
from google.cloud import aiplatform

REGION = "us-central1"

aiplatform.init(
    project=PROJECT_ID,
    location=REGION
)

print("Vertex AI initialized successfully")

[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].
Operation "operations/acat.p2-933786093071-e276a3e1-1c87-4980-9fbb-469dd62ed406" finished successfully.
Vertex AI initialized successfully


**2.3. Upload plan based on batch**

In [ ]:
for i in range(0, 25):
  upload_batch(i)

Checking batch 0: rows 0 to 1000
Valid plans in batch 0: 1000
Encoding batch 0


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

W0520 09:20:07.574000 6144 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Encoding completed in 20.45 seconds
Writing batch_000.json
Uploading batch_000.json to GCS
Copying file://batch_000.json [Content-Type=application/json]...
/ [0 files][    0.0 B/ 16.1 MiB]                                                
-
- [1 files][ 16.1 MiB/ 16.1 MiB]                                                
Operation completed over 1 objects/16.1 MiB.                                     

Uploading manifest_000.json to GCS
Copying file://manifest_000.json [Content-Type=application/json]...
/ [0 files][    0.0 B/  356.0 B]                                                
/ [1 files][  356.0 B/  356.0 B]                                                
Operation completed over 1 objects/356.0 B.                                      

Batch 0 uploaded successfully
Checking batch 1: rows 1000 to 2000
Valid plans in batch 1: 1000
Encoding batch 1


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

...

Encoding completed in 11.08 seconds
Writing batch_022.json
Uploading batch_022.json to GCS
Copying file://batch_022.json [Content-Type=application/json]...
/ [0 files][    0.0 B/ 16.8 MiB]                                                
-
- [1 files][ 16.8 MiB/ 16.8 MiB]                                                
Operation completed over 1 objects/16.8 MiB.                                     

Uploading manifest_022.json to GCS
Copying file://manifest_022.json [Content-Type=application/json]...
/ [0 files][    0.0 B/  362.0 B]                                                
/ [1 files][  362.0 B/  362.0 B]                                                
Operation completed over 1 objects/362.0 B.                                      

Batch 22 uploaded successfully
Checking batch 23: rows 23000 to 24000
Valid plans in batch 23: 1000
Encoding batch 23


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Encoding completed in 9.13 seconds
Writing batch_023.json
Uploading batch_023.json to GCS
Copying file://batch_023.json [Content-Type=application/json]...
/ [0 files][    0.0 B/ 16.4 MiB]                                                
-
- [1 files][ 16.4 MiB/ 16.4 MiB]                                                
Operation completed over 1 objects/16.4 MiB.                                     

Uploading manifest_023.json to GCS
Copying file://manifest_023.json [Content-Type=application/json]...
/ [0 files][    0.0 B/  362.0 B]                                                
/ [1 files][  362.0 B/  362.0 B]                                                
Operation completed over 1 objects/362.0 B.                                      

Batch 23 uploaded successfully
Checking batch 24: rows 24000 to 25000
Valid plans in batch 24: 1000
Encoding batch 24


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Encoding completed in 6.94 seconds
Writing batch_024.json
Uploading batch_024.json to GCS
Copying file://batch_024.json [Content-Type=application/json]...
/ [0 files][    0.0 B/ 16.4 MiB]                                                
-
- [1 files][ 16.4 MiB/ 16.4 MiB]                                                
Operation completed over 1 objects/16.4 MiB.                                     

Uploading manifest_024.json to GCS
Copying file://manifest_024.json [Content-Type=application/json]...
/ [0 files][    0.0 B/  362.0 B]                                                
/ [1 files][  362.0 B/  362.0 B]                                                
Operation completed over 1 objects/362.0 B.                                      

Batch 24 uploaded successfully


**2.4. Save lookup file**

In [ ]:
# =========================================
# CELL 1: SAVE FULL CLEANED DATA TO JSON
# =========================================

import json
from pathlib import Path


OUTPUT_DIR = "outputs"
OUTPUT_JSON = f"{OUTPUT_DIR}/plan_lookup.json"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


def safe_json_value(value):
    """
    Convert unsupported values into JSON-safe values
    """

    try:
        json.dumps(value)
        return value

    except:
        return str(value)


plan_lookup = {}

print("Building plan lookup...")

for idx, row in enumerate(data):
    plan_id = str(row.get("plan_id", f"row_{idx}"))

    cleaned_row = {
        key: safe_json_value(value)
        for key, value in row.items()
    }

    plan_lookup[plan_id] = cleaned_row

    if (idx + 1) % 1000 == 0:
        print(f"Processed {idx + 1} plans")


print("Saving JSON file...")

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(
        plan_lookup,
        f,
        ensure_ascii=False
    )

print("Saved successfully")
print("Output:", OUTPUT_JSON)
print("Total plans:", len(plan_lookup))

Building plan lookup...
Processed 1000 plans
Processed 2000 plans
Processed 3000 plans
Processed 4000 plans
Processed 5000 plans
Processed 6000 plans
Processed 7000 plans
Processed 8000 plans
Processed 9000 plans
Processed 10000 plans
Processed 11000 plans
Processed 12000 plans
Processed 13000 plans
Processed 14000 plans
Processed 15000 plans
Processed 16000 plans
Processed 17000 plans
Processed 18000 plans
Processed 19000 plans
Processed 20000 plans
Processed 21000 plans
Processed 22000 plans
Processed 23000 plans
Processed 24000 plans
Processed 25000 plans
Saving JSON file...
Saved successfully
Output: outputs/plan_lookup.json
Total plans: 25670


In [ ]:
# =========================================
# CELL 2: UPLOAD plan_lookup.json TO GCS
# =========================================

import subprocess


BUCKET = "energy-plan-bucket-1"
GCS_PREFIX = "gte-modernbert-processed-plans"

LOCAL_FILE = "outputs/plan_lookup.json"

GCS_PATH = (
    f"gs://{BUCKET}/"
    f"{GCS_PREFIX}/plan_store/plan_lookup.json"
)


def run_cmd(cmd):
    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.stdout:
        print(result.stdout)

    if result.stderr:
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed: {' '.join(cmd)}"
        )


print("Uploading to GCS...")
print("Local:", LOCAL_FILE)
print("GCS:", GCS_PATH)

run_cmd([
    "gsutil",
    "cp",
    LOCAL_FILE,
    GCS_PATH
])

print("Upload completed successfully")

Uploading to GCS...
Local: outputs/plan_lookup.json
GCS: gs://energy-plan-bucket-1/gte-modernbert-processed-plans/plan_store/plan_lookup.json
Copying file://outputs/plan_lookup.json [Content-Type=application/json]...
/ [0 files][    0.0 B/183.2 MiB]                                                
==> NOTE: You are uploading one or more large file(s), which would run
significantly faster if you enable parallel composite uploads. This
feature can be enabled by editing the
"parallel_composite_upload_threshold" value in your .boto
configuration file. However, note that if you do this large files will
be uploaded as `composite objects
<https://cloud.google.com/storage/docs/composite-objects>`_,which
means that any user who downloads such objects will need to have a
compiled crcmod installed (see "gsutil help crcmod"). This is because
without a compiled crcmod, computing checksums on composite objects is
so slow that gsutil disables downloads of composite objects.

-
- [0 files][  9.8 MiB/18

**2.5. CREATE INDEX (PLEASE DONT RUN AUTOMATICCALLY)**

In [ ]:
from google.cloud import aiplatform

BUCKET = "energy-plan-bucket-1"
GCS_PREFIX = "gte-modernbert-processed-plans"
INDEX_DISPLAY_NAME = "energy-plan-gte-modernbert-cosine-tree-ah-index-1"

DIMENSIONS = 768

def create_tree_ah_index():
    contents_uri = f"gs://{BUCKET}/{GCS_PREFIX}/batches"

    print("Creating Tree-AH index")
    print("Source:", contents_uri)
    print("Dimensions:", DIMENSIONS)
    print("Distance measure: COSINE_DISTANCE")

    index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
        display_name=INDEX_DISPLAY_NAME,
        contents_delta_uri=contents_uri,
        dimensions=DIMENSIONS,
        approximate_neighbors_count=100,
        distance_measure_type="COSINE_DISTANCE",
        leaf_node_embedding_count=500,
        leaf_nodes_to_search_percent=7,
    )

    print("Index created successfully")
    print("Index resource name:", index.resource_name)

    return index


def get_or_create_index():
    print("Checking existing indexes")

    existing_indexes = aiplatform.MatchingEngineIndex.list(
        filter=f'display_name="{INDEX_DISPLAY_NAME}"'
    )

    if existing_indexes:
        index = existing_indexes[0]
        print("Index already exists")
        print("Index resource name:", index.resource_name)
        return index

    return create_tree_ah_index()


index = get_or_create_index()

Checking existing indexes
Creating Tree-AH index
Source: gs://energy-plan-bucket-1/gte-modernbert-processed-plans/batches
Dimensions: 768
Distance measure: COSINE_DISTANCE
Index created successfully
Index resource name: projects/933786093071/locations/us-central1/indexes/6704586610689703936


**2.6. CREATE ENDPOINT**

In [ ]:
from google.cloud import aiplatform

PROJECT_ID = "project-ce1ff6dc-7e15-4f39-bb3"
REGION = "us-central1"

aiplatform.init(
    project=PROJECT_ID,
    location=REGION
)

INDEX_RESOURCE_NAME = "projects/933786093071/locations/us-central1/indexes/6704586610689703936"

index = aiplatform.MatchingEngineIndex(
    index_name=INDEX_RESOURCE_NAME
)

endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
    display_name="energy-plan-endpoint",
    public_endpoint_enabled=True
)

print(endpoint.resource_name)

DEPLOYED_INDEX_ID = "energy_plan_cosine_index"

endpoint.deploy_index(
    index=index,
    deployed_index_id=DEPLOYED_INDEX_ID
)

print("Index deployed successfully")

projects/933786093071/locations/us-central1/indexEndpoints/6138012666943242240
Index deployed successfully


**2.7. UNDEPLOY ENDPOINT**

In [ ]:
from google.cloud import aiplatform


PROJECT_ID = "project-ce1ff6dc-7e15-4f39-bb3"
REGION = "us-central1"

ENDPOINT_RESOURCE_NAME = (
    "projects/933786093071/"
    "locations/us-central1/"
    "indexEndpoints/6138012666943242240"
)

DEPLOYED_INDEX_ID = "energy_plan_cosine_index"


aiplatform.init(
    project=PROJECT_ID,
    location=REGION
)


try:
    endpoint = aiplatform.MatchingEngineIndexEndpoint(
        index_endpoint_name=ENDPOINT_RESOURCE_NAME
    )

    print("Undeploying index...")

    endpoint.undeploy_index(
        deployed_index_id=DEPLOYED_INDEX_ID
    )

    print("Index undeployed successfully")

except Exception as e:
    print("Undeploy failed:")
    print(e)


try:
    print("Deleting endpoint...")

    endpoint.delete(force=True)

    print("Endpoint deleted successfully")

except Exception as e:
    print("Delete endpoint failed:")
    print(e)

Undeploying index...
Index undeployed successfully
Deleting endpoint...
Endpoint deleted successfully
